## Implementar Expanding-Window Cross-Validation

Identifiquei um problema: O uso de uma janela estática (single-window) (ex: 70% treino / 15% validação / 15% teste) em séries temporais financeiras não-estacionárias gera resultados ilusórios e overfitting a um regime de mercado específico.

para poder arrumar isso, eu pretendo substituir o particionamento estático por janelas deslizantes em expansão. O modelo treina nos meses $1$ a $t$ e testa no mês $t+1$. Na iteração seguinte, treina nos meses $1$ a $t+1$ e testa no mês $t+2$.

Minha intenção é de simular o fluxo contínuo do tempo real, garantindo que o modelo nunca tem acesso a dados futuros

In [3]:
import numpy as np
import pandas as pd
from pandas.tseries.offsets import BDay
from sklearn.base import BaseEstimator, ClassifierMixin

In [4]:
class MajorityClassifier(BaseEstimator, ClassifierMixin):
    """Sempre prevê a classe mais frequente do conjunto de treino."""
    def fit(self, X, y):
        self.majority_class_ = pd.Series(y).mode()[0]
        return self
    
    def predict(self, X):
        return np.full(len(X), self.majority_class_)

class ProbabilisticClassifier(BaseEstimator, ClassifierMixin):
    """Prevê aleatoriamente seguindo a distribuição das classes do treino."""
    def fit(self, X, y):
        self.probs_ = pd.Series(y).value_counts(normalize=True).sort_index().values
        self.classes_ = np.sort(np.unique(y))
        return self
    
    def predict(self, X):
        return np.random.choice(self.classes_, size=len(X), p=self.probs_)

class InertiaClassifier(BaseEstimator, ClassifierMixin):
    """Prevê que a direção de amanhã será igual à de hoje (Naïve)."""
    def fit(self, X, y=None):
        return self
    
    def predict(self, X):
        # X deve conter o 'lag_1' ou o retorno do dia anterior
        # Se X for um DataFrame com 'target_lag1', usamos ele
        if isinstance(X, pd.DataFrame) and 'target_lag1' in X.columns:
            return X['target_lag1'].values
        else:
            # Fallback: se não tivermos a info, prevemos 0 (ou erro)
            return np.zeros(len(X))

### Preparação dos Dados e Expanding Window

Vamos carregar os dados e definir a lógica de validação temporal.

In [5]:
def load_data():
    df = pd.read_csv('../../V1/2.stocks/dataset_full.csv', parse_dates=['Date'])
    df.set_index('Date', inplace=True)
    
    # Definir target (sobe em 21 dias)
    horizon = 21
    df['target'] = (df['Close'].shift(-horizon) > df['Close']).astype(int)
    
    # Feature para o InertiaClassifier: direção do retorno de hoje (vs 21 dias atrás)
    df['target_lag1'] = (df['Close'] > df['Close'].shift(horizon)).astype(int)
    
    # Remover NaNs (do shift do target e do lag)
    df.dropna(inplace=True)
    return df

df = load_data()
print(f"Dataset carregado: {df.shape[0]} amostras")

Dataset carregado: 1227 amostras


### Correção do Look-Ahead Bias

Notícias publicadas após o fechamento do mercado (18h) não devem ser usadas para prever o fechamento do mesmo dia, pois essa informação não estava disponível para o investidor durante o pregão. Vamos implementar a lógica para deslocar essas notícias para o próximo dia útil.

In [6]:
def apply_market_closure_rule(df_news, closure_hour=18):
    """
    Ajusta a data das notícias baseada no horário de fechamento da B3.
    df_news deve ter uma coluna 'datetime'.
    """
    df = df_news.copy()
    
    # Se a notícia foi publicada após as 18h, ela pertence ao próximo pregão
    mask_after_closure = df['datetime'].dt.hour >= closure_hour
    
    # Ajustar data: +1 dia útil (BDay)
    df.loc[mask_after_closure, 'adjusted_date'] = df.loc[mask_after_closure, 'datetime'].dt.normalize() + BDay(1)
    df.loc[~mask_after_closure, 'adjusted_date'] = df.loc[~mask_after_closure, 'datetime'].dt.normalize()
    
    return df

# Exemplo de simulação do impacto
example_news = pd.DataFrame({
    'datetime': pd.to_datetime([
        '2024-03-15 10:00:00', # Durante o pregão
        '2024-03-15 19:00:00', # Após o pregão (Sexta)
        '2024-03-16 12:00:00', # Sábado
        '2024-03-18 08:00:00'  # Segunda antes do pregão
    ]),
    'sentiment': [0.8, -0.5, 0.2, 0.9]
})

adjusted_example = apply_market_closure_rule(example_news)
adjusted_example[['datetime', 'adjusted_date', 'sentiment']]

,datetime,adjusted_date,sentiment
0,2024-03-15 10:00:00,2024-03-15,0.8
1,2024-03-15 19:00:00,2024-03-18,-0.5
2,2024-03-16 12:00:00,2024-03-16,0.2
3,2024-03-18 08:00:00,2024-03-18,0.9


### Avaliação de Baselines com Expanding Window

Agora vamos executar a validação walk-forward para os nossos modelos ingênuos.

In [7]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def evaluate_expanding_window(df, model, min_train_days=252*2, test_days=21):
    results = []
    n = len(df)
    
    for start in range(min_train_days, n, test_days):
        end_test = min(start + test_days, n)
        if start >= end_test: break
        
        train_df = df.iloc[:start]
        test_df = df.iloc[start:end_test]
        
        X_train, y_train = train_df.drop('target', axis=1), train_df['target']
        X_test, y_test = test_df.drop('target', axis=1), test_df['target']
        
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        
        results.append({
            'y_true': y_test.values,
            'y_pred': preds
        })
        
    return results

def calculate_metrics(eval_results):
    if not eval_results: return {}
    y_true = np.concatenate([r['y_true'] for r in eval_results])
    y_pred = np.concatenate([r['y_pred'] for r in eval_results])
    
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred, average='macro'),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0)
    }

baselines = {
    'Majority': MajorityClassifier(),
    'Probabilistic': ProbabilisticClassifier(),
    'Inertia': InertiaClassifier()
}

metrics = {}
for name, model in baselines.items():
    res = evaluate_expanding_window(df, model)
    metrics[name] = calculate_metrics(res)

pd.DataFrame(metrics).T

,accuracy,f1,precision,recall
Majority,0.618257,0.382051,0.618257,1.000000
Probabilistic,0.507607,0.489820,0.610706,0.561521
Inertia,0.605809,0.575901,0.673077,0.704698


### Análise dos Resultados dos Baselines

Os resultados acima estabelecem o **piso de desempenho** que qualquer modelo de Deep Learning (como o Transformer ou LSTM) deve superar para provar sua utilidade. 

**Observações Principais:**

1. **Dominância da Classe Majoritária:** O baseline *Majority* atinge ~61.8% de acurácia apenas prevendo "Sobe". Isso indica que qualquer modelo que reporte acurácia nessa faixa sem um F1-Score (macro) equilibrado está provavelmente apenas replicando a tendência do mercado, sem poder preditivo real.

2. **A Força da Inércia:** O baseline *Inertia* (que prevê que a direção dos próximos 21 dias será igual à dos últimos 21) atinge ~60.6% de acurácia e um F1 de 0.57. Ele é particularmente forte em Precision (~67.3%), o que sugere que o mercado tem momentos de tendência (momentum) que são capturados por regras simples.

3. **O Desafio do Transformer:** No Capítulo 4, o Transformer reportou 76.3% de acurácia. Agora vemos que o "pulo" real não foi de 50% (acaso) para 76%, mas sim de **62% (maioritário) para 76%**. Embora ainda seja um ganho, a margem é menor e exige validação estatística (como o teste de Wilcoxon mencionado no Capítulo 5) para confirmar se não é ruído.

**Conclusão da Fase 1:**
Com estes baselines e a correção de Look-Ahead Bias, temos agora um ambiente de teste rigoroso. O próximo passo é reavaliar os modelos de sentimento FinBERT dentro deste loop de *Expanding Window* para verificar se o ganho de AUC observado anteriormente se mantém de forma consistente ao longo de diferentes regimes de mercado.